<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m2_eigen_svd_pca_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 · Notebook 2 — Eigen, SVD & PCA
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

A **self-contained** continuation of Notebook 1: the two factorisations that power machine learning — the
**eigendecomposition** and the **singular value decomposition** — and the two payoffs they unlock,
**least squares** (Géron Ch. 4) and **PCA** (Ch. 7). Worked on real biomedical data and a real image.

Run in **Google Colab** (*Runtime → Run all*). Everything is offline (the image and datasets ship with
Scikit-Learn).

**Contents**
1. Eigenvalues and eigenvectors (invariant directions)
2. Diagonalisation
3. Symmetric matrices and the spectral theorem
4. The SVD: rotate–scale–rotate
5. Low-rank image compression (bridge to Module 4)
6. Least squares via the normal equation (Ch. 4)
7. PCA on the breast-cancer data (Ch. 7)

Most sections end with an **Exercise**; run the **Solution** cell to check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
print("ready")

---
## 1 · Eigenvalues and eigenvectors

An **eigenvector** is a direction the matrix only *stretches*, not turns: `A v = lambda v`. We compute
them with `np.linalg.eig`, then *see* them: a circle of vectors maps to an ellipse, and the eigenvectors
are the directions that stay on their own line.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
vals, vecs = np.linalg.eig(A)
print("eigenvalues :", vals)            # 3 and 1
print("eigenvectors (columns):\n", vecs)
# check A v = lambda v for the first eigenpair
print("A v0 =", A @ vecs[:, 0], " vs  lambda0 v0 =", vals[0] * vecs[:, 0])

In [ ]:
# Visualise: unit circle -> ellipse; eigenvectors stay on their line
t = np.linspace(0, 2*np.pi, 200)
circle = np.array([np.cos(t), np.sin(t)])
ellipse = A @ circle
fig, ax = plt.subplots(figsize=(5,5))
ax.plot(*circle, color="lightgray", label="unit circle")
ax.plot(*ellipse, color="steelblue", label="A(circle)")
for i in range(2):
    v = vecs[:, i]
    ax.quiver(0,0,*v, angles="xy",scale_units="xy",scale=1,color="crimson",width=0.012)
    ax.quiver(0,0,*(A@v), angles="xy",scale_units="xy",scale=1,color="darkgreen",width=0.008)
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Eigenvectors (red) map to themselves scaled (green)"); plt.show()

**Exercise 1.** A simple two-compartment drug model transfers concentration between blood and tissue each time step via `A = np.array([[0.9, 0.1], [0.2, 0.8]])`. Find its eigenvalues and eigenvectors, and identify which eigenvalue corresponds to the slower-decaying mode (the one nearer 1) — that is the long-term direction the concentrations settle into.

> 🤖 *Gemini tip:* "Given a 2x2 NumPy matrix that models transfer between two compartments, show me how to find its eigenvalues/eigenvectors with np.linalg.eig and explain what the eigenvalue closest to 1 means physically."

In [ ]:
Acomp = np.array([[0.9, 0.1], [0.2, 0.8]])

# Your code here


---
## 2 · Diagonalisation

If `A` has independent eigenvectors, `A = P @ diag(lambda) @ inv(P)`: in the eigenvector basis the map is
pure scaling. This makes matrix powers trivial.

In [ ]:
P = vecs
Lam = np.diag(vals)
A_rebuilt = P @ Lam @ np.linalg.inv(P)
print("reconstructed A:\n", A_rebuilt, "\nmatches:", np.allclose(A, A_rebuilt))
# powers via diagonalisation: A^5
A5 = P @ np.diag(vals**5) @ np.linalg.inv(P)
print("A^5 matches np.linalg.matrix_power:", np.allclose(A5, np.linalg.matrix_power(A, 5)))

**Exercise 2.** Using the same compartment matrix `Acomp` from Exercise 1, use diagonalisation to predict the concentration state after 20 time steps starting from an initial state `x0 = np.array([1.0, 0.0])` (all drug starts in blood): compute `A^20 @ x0` via `P @ diag(lambda^20) @ inv(P) @ x0`, and confirm it matches `np.linalg.matrix_power(A, 20) @ x0`.

> 🤖 *Gemini tip:* "Show me how to use an eigendecomposition to compute A to a high power applied to a starting vector, and confirm it matches np.linalg.matrix_power."

In [ ]:
Acomp = np.array([[0.9, 0.1], [0.2, 0.8]])   # same compartment matrix as Exercise 1
x0 = np.array([1.0, 0.0])

# Your code here


---
## 3 · Symmetric matrices and the spectral theorem

If `A` is symmetric, its eigenvalues are real and its eigenvectors are **orthonormal** — use `eigh` (faster
and more accurate than `eig` for this case). Covariance matrices are symmetric, which is why PCA's axes
come out orthogonal.

In [ ]:
S = np.array([[2.0, 1.0], [1.0, 2.0]])     # symmetric
w, Q = np.linalg.eigh(S)
print("eigenvalues:", w)
print("eigenvectors orthonormal? Q.T@Q = I:", np.allclose(Q.T @ Q, np.eye(2)))
print("dot of the two eigenvectors (should be 0):", round(Q[:,0] @ Q[:,1], 6))

**Exercise 3.** For `S = [[4,1],[1,4]]`, find its eigenvalues with `eigh`, confirm the eigenvectors are
orthogonal, and reconstruct `S` as `Q @ diag(w) @ Q.T`.

> 🤖 *Gemini tip:* "Given a symmetric NumPy matrix, show me how to get its eigenvalues and orthonormal eigenvectors with eigh, and how to check the reconstruction Q @ diag(w) @ Q.T."

In [ ]:
S2 = np.array([[4.0,1.0],[1.0,4.0]])

# Your code here


---
## 4 · The SVD: rotate–scale–rotate

The **singular value decomposition** `A = U @ diag(S) @ Vt` works for *any* matrix. Geometrically it is a
rotation, an axis-aligned scaling by the **singular values**, then another rotation — so the unit circle
maps to an ellipse whose semi-axis lengths are the singular values.

In [ ]:
M = np.array([[2.0, 1.0], [0.0, 1.5]])
U, S, Vt = np.linalg.svd(M)
print("singular values:", S)
print("U orthogonal:", np.allclose(U.T@U, np.eye(2)), "| V orthogonal:", np.allclose(Vt@Vt.T, np.eye(2)))
ell = M @ circle
fig, ax = plt.subplots(figsize=(5,5))
ax.plot(*circle, color="lightgray", label="unit circle")
ax.plot(*ell, color="steelblue", label="M(circle)")
for i in range(2):                              # semi-axes = sigma_i * u_i
    ax.quiver(0,0,*(S[i]*U[:,i]), angles="xy",scale_units="xy",scale=1,color="crimson",width=0.012)
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Singular values = ellipse semi-axis lengths"); plt.show()

**Exercise 4.** A 3-electrode EMG headset mixes two underlying muscle-activity sources through `M = np.array([[0.9, 0.2], [0.1, 0.8], [0.5, 0.5]])` (a 3x2 "mixing" matrix, one row per electrode). Compute its SVD and report the singular values — the larger one tells you the dominant, most energetic mixing direction.

> 🤖 *Gemini tip:* "Given a non-square NumPy matrix that mixes two signal sources into several sensor channels, show me how to compute its SVD and interpret the singular values."

In [ ]:
Memg = np.array([[0.9, 0.2], [0.1, 0.8], [0.5, 0.5]])

# Your code here


---
## 5 · Low-rank image compression (bridge to Module 4)

Keeping only the largest `k` singular values gives the **best** rank-`k` approximation of a matrix
(Eckart–Young). An image is just a matrix of pixel intensities, so SVD compresses it: most of the picture
lives in a few singular values. (Here a sample photo that ships with Scikit-Learn; in Module 4 you will do
the same on **Broad BBBC microscopy** images.)

In [ ]:
from sklearn.datasets import load_sample_image
img = load_sample_image("china.jpg").astype(float)
gray = img @ np.array([0.2989, 0.587, 0.114])    # RGB -> grayscale (a 2-D matrix)
print("image matrix shape:", gray.shape)

U, S, Vt = np.linalg.svd(gray, full_matrices=False)
def rank_k(k):
    return (U[:, :k] * S[:k]) @ Vt[:k, :]

ks = [5, 20, 50, len(S)]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, k in zip(axes, ks):
    ax.imshow(rank_k(k), cmap="gray")
    stored = k*(gray.shape[0] + gray.shape[1] + 1)
    ratio = gray.size / stored
    ax.set_title(f"k={k}  ({ratio:.1f}x smaller)" if k < len(S) else f"k={k} (full)")
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# How much 'energy' (variance) the top-k singular values capture
energy = np.cumsum(S**2) / np.sum(S**2)
fig, ax = plt.subplots(figsize=(6,3))
ax.plot(range(1, 61), energy[:60], marker=".")
ax.axhline(0.95, ls="--", color="gray")
ax.set_xlabel("number of singular values k"); ax.set_ylabel("fraction of energy")
ax.set_title("A few components capture most of the image"); ax.grid(alpha=0.3); plt.show()
print("k for 95% energy:", int(np.argmax(energy >= 0.95)) + 1)

**Exercise 5.** Using the `gray` image and its SVD from above, compute the reconstruction error (Frobenius norm of the difference) between the rank-`k=20` approximation and the original image, as a *fraction* of the image's own norm. Then find the smallest `k` for which this relative error drops below 5%.

> 🤖 *Gemini tip:* "Given the SVD of an image matrix in NumPy, show me how to compute the relative Frobenius-norm reconstruction error of a rank-k approximation, and how to search for the smallest k that keeps that error under a target threshold."

In [ ]:
# Your code here


---
## 6 · Least squares via the normal equation (Géron Ch. 4)

Fitting `X w ≈ y` with more rows than unknowns is a **projection**: choose `w` so the residual is
orthogonal to every column of `X`, giving the **normal equation** `(X.T X) w = X.T y`. This is the
closed-form linear regression of Chapter 4. We solve it three ways and check they agree.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
d = load_diabetes()
X = np.c_[np.ones(len(d.data)), d.data]    # add a bias column
y = d.target

# (a) normal equation
w_normal = np.linalg.inv(X.T @ X) @ X.T @ y
# (b) np.linalg.lstsq (SVD-based, numerically safer)
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
# (c) scikit-learn
lr = LinearRegression(fit_intercept=False).fit(X, y)

print("normal vs lstsq agree:", np.allclose(w_normal, w_lstsq))
print("lstsq vs sklearn agree:", np.allclose(w_lstsq, lr.coef_))
print("first 3 coefficients:", w_normal[:3])

**Exercise 6.** Compute the model's predictions `yhat = X @ w_normal` and report the R² score
`1 - SS_res/SS_tot`. Confirm it matches `lr.score(X, y)`.

> 🤖 *Gemini tip:* "Show me how to compute the R-squared score for a linear regression by hand from predictions and true values, and how it compares to scikit-learn's .score()."

In [ ]:
# Your code here


---
## 7 · PCA on the breast-cancer data (Géron Ch. 7)

**PCA** rotates the feature axes to an orthonormal basis ordered by variance — a change of basis found by
the SVD. We standardise the 30 features, take the SVD, project onto the top 2 components, and see the
classes separate. Then we confirm it matches Scikit-Learn's `PCA`.

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer()
Xs = (bc.data - bc.data.mean(0)) / bc.data.std(0)     # standardise (Module 1)

U, S, Vt = np.linalg.svd(Xs, full_matrices=False)
scores = U[:, :2] * S[:2]                              # 2-D PCA projection
explained = (S**2 / np.sum(S**2))
print("variance explained by PC1, PC2:", explained[:2].round(3),
      "| cumulative:", explained[:2].sum().round(3))

fig, ax = plt.subplots(figsize=(6,5))
for cls, name, col in [(0,"malignant","crimson"), (1,"benign","steelblue")]:
    m = bc.target == cls
    ax.scatter(scores[m,0], scores[m,1], s=12, alpha=0.6, color=col, label=name)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend()
ax.set_title("Breast-cancer data in its first two principal components"); plt.show()

In [ ]:
# Confirm against scikit-learn (signs of components are arbitrary, so compare magnitudes)
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(Xs)
print("sklearn explained variance ratio:", pca.explained_variance_ratio_.round(3))
print("matches our SVD values:", np.allclose(pca.explained_variance_ratio_, explained[:2]))

**Exercise 7.** How many principal components are needed to capture at least 90% of the variance in
the standardised breast-cancer features? *Hint:* cumulative sum of `explained`.

> 🤖 *Gemini tip:* "Given an array of per-component explained-variance ratios from PCA, show me how to find the smallest number of components whose cumulative variance reaches a target threshold."

In [ ]:
# Your code here


---
### Module 2 complete
Eigenvectors and diagonalisation, the spectral theorem, the SVD and low-rank approximation, least squares
as projection, and PCA as an SVD change-of-basis — the linear algebra Géron's Chapters 4 and 7 run on, and
the $SO(3)$/rotation algebra shared with the Robotics unit. **Next:** Module 3 (Calculus) turns to
derivatives, gradients and the integral behind the PID controller.